# Exploración de los datos de OT Consulting
Reto: Soporte analítico para la toma de decisiones, Etapa 2

Javier Barrón Vargas, A00842507

## Cargar los archivos

Aquí estoy importando las librerías y cargando los diez Excel que exporté de la aplicación. Para que funcione en Colab hay que subir los archivos en el panel de la izquierda.

In [ ]:
import pandas as pd

colaboradores = pd.read_excel('Catalogo_Colaboradores.xlsx')
sueldos       = pd.read_excel('Sueldos_Salarios_OTC.xlsx')
roles         = pd.read_excel('Catalogo_Roles.xlsx')
departamentos = pd.read_excel('Catalogo_Departamentos.xlsx')
habilidades   = pd.read_excel('Catalogo_Habilidades.xlsx')
matriz        = pd.read_excel('Matriz_Habilidades.xlsx')
clientes      = pd.read_excel('Catalogo_Clientes.xlsx')
asignaciones  = pd.read_excel('Asignaciones_Colaboradores.xlsx')
activos       = pd.read_excel('Catalogo_Activos.xlsx')
inversion     = pd.read_excel('Inversion_Activos_OTC.xlsx')

archivos = {'Colaboradores': colaboradores, 'Sueldos': sueldos, 'Roles': roles,
            'Departamentos': departamentos, 'Habilidades': habilidades, 'Matriz': matriz,
            'Clientes': clientes, 'Asignaciones': asignaciones, 'Activos': activos,
            'Inversion': inversion}

for nombre, df in archivos.items():
    print(nombre, '->', df.shape[0], 'filas y', df.shape[1], 'columnas')

Colaboradores -> 107 filas y 45 columnas
Sueldos -> 75 filas y 8 columnas
Roles -> 55 filas y 6 columnas
Departamentos -> 6 filas y 7 columnas
Habilidades -> 71 filas y 6 columnas
Matriz -> 54 filas y 77 columnas
Clientes -> 12 filas y 13 columnas
Asignaciones -> 31 filas y 11 columnas
Activos -> 123 filas y 11 columnas
Inversion -> 123 filas y 7 columnas


## Revisar las columnas de cada archivo

Aquí hice una función para revisar todas las columnas de un archivo de una vez. Me dice qué tipo de dato tiene cada columna, cuántos vacíos hay, cuántos dicen PENDIENTE y cuántos valores distintos tiene.

In [ ]:
def revisar(df):
    resumen = pd.DataFrame({
        'tipo': df.dtypes.astype(str),
        'vacios': df.isna().sum(),
        'pendientes': (df.astype(str).apply(lambda col: col.str.upper().str.strip()) == 'PENDIENTE').sum(),
        'valores_distintos': df.nunique(),
    })
    return resumen

## Colaboradores

Aquí estoy revisando el catálogo de colaboradores, que es la tabla principal.

In [ ]:
revisar(colaboradores)

,tipo,vacios,pendientes,valores_distintos
ID,object,0,0,107
Nombre Completo,object,0,0,107
Género,object,0,0,2
Fecha Nacimiento,object,1,0,104
Teléfono,int64,0,0,1
Correo Electrónico,object,0,0,1
Ciudad,object,0,0,25
Domicilio,object,0,0,1
RFC,object,0,0,1
CURP,object,0,0,1


Aquí estoy comprobando mi primer hallazgo: buscar a los colaboradores activos que tienen el puesto en pendiente y no tienen departamento, y ver si tienen sueldo.

In [ ]:
activos_colab = colaboradores[colaboradores['Estatus'] == 'Activo']
sin_puesto = activos_colab[(activos_colab['Puesto'] == 'PENDIENTE') & (activos_colab['Departamento'].isna())]

print('Colaboradores activos:', len(activos_colab))
print('Activos sin puesto ni departamento:', len(sin_puesto))
print('De esos, cuántos tienen sueldo:', (sin_puesto['Sueldo Neto Total'] > 0).sum())
print('Nómina mensual de esas personas:', round(sin_puesto['Sueldo Total Integrado'].sum(), 2))

Colaboradores activos: 75
Activos sin puesto ni departamento: 29
De esos, cuántos tienen sueldo: 27
Nómina mensual de esas personas: 1691904.92


Aquí estoy revisando si las bajas tienen su fecha de baja y cómo se escriben las ciudades y los seguros médicos.

In [ ]:
bajas = colaboradores[colaboradores['Estatus'] == 'Baja']
print('Bajas:', len(bajas), '| Bajas sin fecha de baja:', bajas['Fecha de Baja'].isna().sum())
print()
print(colaboradores['Ciudad'].value_counts().head(10))
print()
print(colaboradores['SGMM'].value_counts())

Bajas: 32 | Bajas sin fecha de baja: 18

Ciudad
Monterrey         50
Edo de México     12
CDMX              12
Puebla             7
Tlaxcala           4
Queretaro          2
Cuernavaca         2
Guadalajara        1
Toluca             1
Ciudad Obregón     1
Name: count, dtype: int64

SGMM
METLIFE             36
SEGUROS ATLAS       24
PLAN SEGURO         15
NO                  11
SI                   6
PLAN SEGUROS         4
PLAN ATLAS           2
PENDIENTE            1
NO LO ENVIO          1
PENDIENTE LLAMAR     1
Name: count, dtype: int64


Aquí estoy comprobando si las columnas de sueldo cuadran entre sí.

In [ ]:
suma = colaboradores['Sueldo Neto IMSS'] + colaboradores['Sueldo Asimilados/Sindicato'] + colaboradores['Sueldo Complementos']
print('Registros donde no cuadra el sueldo neto:', ((suma - colaboradores['Sueldo Neto Total']).abs() > 1).sum())
print('Activos con sueldo en cero:', (activos_colab['Sueldo Neto Total'] == 0).sum())

Registros donde no cuadra el sueldo neto: 0
Activos con sueldo en cero: 5


Con esto se confirma mi hallazgo: hay 29 colaboradores activos sin puesto ni departamento y 27 de ellos sí cobran, lo que da casi 1.7 millones de pesos al mes que no se sabe a qué área cargar. Además encontré que 18 bajas no tienen fecha de baja y que las ciudades y los seguros se escriben de diferentes formas, por ejemplo Monterrey y MTY, o PLAN SEGURO y PLAN SEGUROS. Lo bueno es que las fórmulas de sueldo sí cuadran.

## Sueldos

Aquí estoy revisando el archivo de sueldos y comparándolo con el catálogo de colaboradores.

In [ ]:
revisar(sueldos)

,tipo,vacios,pendientes,valores_distintos
ID,object,0,0,75
Nombre Colaborador,object,0,0,75
Estatus,object,0,0,1
Tipo de Nómina,object,0,0,8
Departamento,object,64,0,2
Puesto/Rol,object,0,29,23
Sueldo Neto Total,float64,0,0,63
Sueldo Total Integrado,float64,0,0,67


In [ ]:
comparacion = sueldos.merge(colaboradores, on='ID')
print('Registros en sueldos:', len(sueldos))
print('Sueldos que no coinciden con el catálogo:', (comparacion['Sueldo Neto Total_x'] != comparacion['Sueldo Neto Total_y']).sum())

Registros en sueldos: 75
Sueldos que no coinciden con el catálogo: 0


Este archivo tiene a los 75 colaboradores activos con los mismos sueldos del catálogo, así que no tiene errores pero tampoco agrega información nueva.

## Roles

Aquí estoy revisando el catálogo de roles.

In [ ]:
revisar(roles)

,tipo,vacios,pendientes,valores_distintos
Código,object,0,0,55
Nombre del Rol,object,0,1,55
Nivel,object,0,0,7
Departamento,object,2,0,5
Descripción,object,0,1,55
Estatus,object,0,0,1


In [ ]:
print('Roles donde la descripción es igual al nombre:', (roles['Descripción'].str.upper() == roles['Nombre del Rol'].str.upper()).sum(), 'de', len(roles))
roles[roles['Departamento'].isna()]

Roles donde la descripción es igual al nombre: 55 de 55


,Código,Nombre del Rol,Nivel,Departamento,Descripción,Estatus
49,R0137,FAMILIAR COLABORADOR,Practicante,NaN,FAMILIAR COLABORADOR,Activo
54,R0142,PENDIENTE,Jefe de Área,NaN,PENDIENTE,Activo


En todos los roles la descripción es igual al nombre, entonces no explica nada del puesto. También existe un rol que se llama PENDIENTE y que se usa como relleno. Además el catálogo no tiene la tarifa por hora de cada rol, que es algo que se va a necesitar para calcular cuánto gana la empresa.

## Departamentos

Aquí estoy revisando los departamentos para comprobar lo del centro de costos y el responsable.

In [ ]:
display(revisar(departamentos))
departamentos

,tipo,vacios,pendientes,valores_distintos
uid,int64,0,0,6
objetivo,object,0,0,6
estatus,object,0,0,1
responsable,float64,6,0,0
nombre,object,0,0,6
centroCostos,float64,6,0,0
codigo,object,0,0,6


,uid,objetivo,estatus,responsable,nombre,centroCostos,codigo
0,1789623822518,"Conseguir nuevos clientes, dar seguimiento a l...",Activo,NaN,Ventas,NaN,D0001
1,1789623978382,Tener el control de los ingresos y egresos de ...,Activo,NaN,Administración y Finanzas,NaN,D0002
2,1789624127466,"Reclutamiento y selección del personal, admini...",Activo,NaN,Recursos Humanos,NaN,D0003
3,1789624282795,"Control y seguimiento a los proyectos, adminis...",Activo,NaN,Operaciones y Soporte,NaN,D0004
4,1789624377678,Administración general de OTC,Activo,NaN,Dirección General,NaN,D0005
5,1789665380680,Administrar la documentación y automatización ...,Activo,NaN,Procesos y Mejora Continua,NaN,D0006


Se confirma que ninguno de los seis departamentos tiene centro de costos ni responsable. Sin el centro de costos no se puede saber cuánto gasta cada área.

## Habilidades y matriz de habilidades

Aquí estoy revisando las columnas del catálogo de habilidades y de la matriz.

In [ ]:
revisar(habilidades)

,tipo,vacios,pendientes,valores_distintos
tipo,object,0,0,2
estatus,object,0,0,1
tecnologia,object,0,0,6
descripcion,object,0,0,58
uid,int64,0,0,71
codigo,object,0,0,71


In [ ]:
display(pd.crosstab(habilidades['tecnologia'], habilidades['tipo']))
revisar(matriz)

tipo,HARD,SOFT
tecnologia,,
Certificaciones,0,1
Idiomas,0,1
Oracle Fusion Cloud,17,0
Oracle eBS,30,0
Otros,0,2
Plataformas Tecnológicas,20,0


,tipo,vacios,pendientes,valores_distintos
ID Colaborador,object,0,0,54
Nombre Completo,object,0,0,54
Correo Electrónico,object,0,0,1
Puesto/Rol,object,0,28,16
Departamento,object,51,0,1
...,...,...,...,...
S0001 - NIVEL DE INGLES,object,0,0,4
S0002 - CERTIFICADO ORACLE,object,42,0,1
H0068 - Oracle EBS,object,13,0,4
S0004 - Empatia,float64,54,0,0


Aquí estoy comprobando mi hallazgo de la matriz: buscar colaboradores con el rol en pendiente que sí tienen habilidades.

In [ ]:
columnas_hab = matriz.columns[6:]
matriz['num_habilidades'] = matriz[columnas_hab].notna().sum(axis=1)

pendientes_con_hab = matriz[(matriz['Puesto/Rol'] == 'PENDIENTE') & (matriz['num_habilidades'] > 0)]
print('Colaboradores en la matriz:', len(matriz))
print('Con rol pendiente pero con habilidades:', len(pendientes_con_hab))
print('Colaboradores activos que no están en la matriz:', len(set(activos_colab['ID']) - set(matriz['ID Colaborador'])))

Colaboradores en la matriz: 54
Con rol pendiente pero con habilidades: 28
Colaboradores activos que no están en la matriz: 28


In [ ]:
matriz[['ID Colaborador', 'Puesto/Rol', 'num_habilidades']].sort_values('num_habilidades', ascending=False).head(10)

,ID Colaborador,Puesto/Rol,num_habilidades
27,OTC048,PENDIENTE,69
15,OTC031,PENDIENTE,68
42,OTC083,CONSULTOR TÉCNICO ORACLE EBS Y CLOUD SENIOR,68
49,OTC095,GERENTE DE PROYECTOS Y OPERACIONES,68
48,OTC094,CONSULTOR ORACLE CLOUD Y EBS SENIOR,68
21,OTC041,PENDIENTE,68
18,OTC037,CONSULTOR TÉCNICO ORACLE EBS Y CLOUD SENIOR,67
12,OTC026,CONSULTOR TÉCNICO ORACLE CLOUD SENIOR,59
0,OTC003,PENDIENTE,51
52,OTC101,CONSULTOR TÉCNICO ORACLE EBS SENIOR,51


Se confirma que hay 28 personas con rol pendiente que sí tienen habilidades, lo que se podría aprovechar para saber qué puesto les toca. También vi que 28 activos no aparecen en la matriz y que hay personas con casi todas las habilidades marcadas, la mayoría en nivel básico, lo que parece que se llenó por default.

## Clientes y proyectos

Aquí estoy revisando el catálogo de clientes y sus proyectos.

In [ ]:
revisar(clientes)

,tipo,vacios,pendientes,valores_distintos
Cód Cliente,object,0,0,8
Cliente,object,0,0,8
Industria,object,0,0,5
RFC,object,8,0,4
Contacto Principal,object,0,0,8
Email Contacto,object,3,0,7
Estatus Cliente,object,0,0,1
Proyecto,object,0,0,12
Fecha Inicio,object,6,0,6
Fecha Fin,object,9,0,3


In [ ]:
clientes[['Cód Cliente', 'Cliente', 'Proyecto', 'Fecha Inicio', 'Fecha Fin', 'Estatus Proyecto']]

,Cód Cliente,Cliente,Proyecto,Fecha Inicio,Fecha Fin,Estatus Proyecto
0,C0001,METALSA,AMS Manufactura,NaN,NaN,En Curso
1,C0001,METALSA,Soporte AMS Manufacta,NaN,NaN,En Curso
2,C0001,METALSA,Asignación indefinida,NaN,NaN,Por Iniciar
3,C0002,Grupo Daltile,Marazzi,2026-07-15,2026-09-23,En Curso
4,C0002,Grupo Daltile,Asignación EBS temporal (3 meses),NaN,NaN,Por Iniciar
5,C0002,Grupo Daltile,Asignación Funcional-Técnico (4 meses),NaN,NaN,Por Iniciar
6,C0003,Allegion,Migración eBs,2026-10-13,2028-10-30,En Curso
7,C0004,CITI,Soporte Oracle Cloud,2025-06-01,NaN,En Curso
8,C0005,Parker Servicios de México,Desarrollo de Indicadores,NaN,NaN,Por Iniciar
9,C0006,Rent A Center (RAC Mèxico Operaciones),SHINE - Asiganci´ñon de recursos de apoyo,2026-06-30,2026-10-30,En Curso


Aquí se ve lo de los proyectos: no tienen un código propio, solo el código del cliente se repite en cada renglón, así que la única forma de distinguir un proyecto de otro es por su nombre. La mitad tampoco tiene fechas, hay nombres mal escritos como Soporte AMS Manufacta, y ningún proyecto tiene monto, tarifa ni horas.

## Asignaciones

Aquí estoy revisando las asignaciones para comprobar lo de los roles y las fechas.

In [ ]:
revisar(asignaciones)

,tipo,vacios,pendientes,valores_distintos
ID Colaborador,object,0,0,30
Nombre Colaborador,object,0,0,30
Nómina/Rol Interno,object,0,14,11
Estatus Colaborador,object,0,0,1
Cliente,object,0,0,3
Proyecto,object,0,0,4
Rol Asignado,object,0,29,2
Tipo de Asignación,object,0,0,2
Estatus Asignación,object,0,0,1
Fecha Inicio,object,0,0,4


In [ ]:
print('Asignaciones con rol PENDIENTE:', (asignaciones['Rol Asignado'] == 'PENDIENTE').sum(), 'de', len(asignaciones))
print()
print('Fechas de inicio más comunes:')
print(asignaciones['Fecha Inicio'].astype(str).value_counts().head(3))
print()
print('Fechas de fin más comunes:')
print(asignaciones['Fecha Fin'].astype(str).value_counts().head(3))

Asignaciones con rol PENDIENTE: 29 de 31

Fechas de inicio más comunes:
Fecha Inicio
1990-01-01    21
2026-10-13     5
2026-01-01     3
Name: count, dtype: int64

Fechas de fin más comunes:
Fecha Fin
Indefinido    24
2027-01-01     3
2027-07-01     3
Name: count, dtype: int64


Aquí estoy cruzando los proyectos con las asignaciones para ver cuántas personas tiene cada proyecto.

In [ ]:
clientes['clave'] = clientes['Cliente'].str.strip() + ' | ' + clientes['Proyecto'].str.strip()
asignaciones['clave'] = asignaciones['Cliente'].str.strip() + ' | ' + asignaciones['Proyecto'].str.strip()

clientes['personas_asignadas'] = clientes['clave'].map(asignaciones['clave'].value_counts()).fillna(0).astype(int)
clientes[['Cliente', 'Proyecto', 'Estatus Proyecto', 'personas_asignadas']]

,Cliente,Proyecto,Estatus Proyecto,personas_asignadas
0,METALSA,AMS Manufactura,En Curso,0
1,METALSA,Soporte AMS Manufacta,En Curso,5
2,METALSA,Asignación indefinida,Por Iniciar,18
3,Grupo Daltile,Marazzi,En Curso,3
4,Grupo Daltile,Asignación EBS temporal (3 meses),Por Iniciar,0
5,Grupo Daltile,Asignación Funcional-Técnico (4 meses),Por Iniciar,0
6,Allegion,Migración eBs,En Curso,5
7,CITI,Soporte Oracle Cloud,En Curso,0
8,Parker Servicios de México,Desarrollo de Indicadores,Por Iniciar,0
9,Rent A Center (RAC Mèxico Operaciones),SHINE - Asiganci´ñon de recursos de apoyo,En Curso,0


In [ ]:
sin_asignacion = set(activos_colab['ID']) - set(asignaciones['ID Colaborador'])
print('Colaboradores activos que no están en ninguna asignación:', len(sin_asignacion))

Colaboradores activos que no están en ninguna asignación: 45


Se confirma todo lo que vi en la página. 29 de las 31 asignaciones tienen el rol en pendiente, 21 empiezan en 1990, que obviamente es una fecha de relleno, y 24 tienen como fecha de fin la palabra Indefinido. También hay proyectos en curso sin nadie asignado, como CITI, Fanosa o Rent A Center, y pasa al revés con Asignación indefinida de METALSA, que dice Por Iniciar pero tiene 18 personas. Además, 45 de los 75 activos no están en ninguna asignación.

## Activos e inversión

Aquí estoy revisando las columnas del inventario de equipos y del archivo de inversión.

In [ ]:
display(revisar(activos))
revisar(inversion)

,tipo,vacios,pendientes,valores_distintos
Cód Activo,object,0,0,123
Descripción,object,0,0,2
Marca,object,0,0,12
Modelo,object,1,0,78
Serie,object,0,1,123
Tipo,object,0,0,2
Estatus,object,0,0,1
Costo,float64,0,0,31
Colaborador Asignado,object,0,0,74
Fecha Inicio,object,32,0,48


,tipo,vacios,pendientes,valores_distintos
Código del Activo,object,0,0,123
Descripción del Producto,object,0,0,2
Estatus,object,0,0,1
Tipo,object,0,0,2
Marca,object,0,0,12
Modelo,object,1,0,78
Costo,float64,0,0,31


In [ ]:
print('Columnas del inventario:', list(activos.columns))
print()
print('Equipos con costo en cero:', (inversion['Costo'] == 0).sum(), 'de', len(inversion))
print('Inversión total registrada:', inversion['Costo'].sum())
print()
print(activos['Marca'].value_counts())

Columnas del inventario: ['Cód Activo', 'Descripción', 'Marca', 'Modelo', 'Serie', 'Tipo', 'Estatus', 'Costo', 'Colaborador Asignado', 'Fecha Inicio', 'Fecha Fin']

Equipos con costo en cero: 70 de 123
Inversión total registrada: 567915.55

Marca
DELL       54
HP         23
LENOVO     17
ACER        8
Lenovo      5
DELL        4
Dell        3
HP          3
LENOVO      2
ASUS        2
Acer        1
Dell        1
Name: count, dtype: int64


Aquí estoy revisando si hay laptops asignadas a personas que ya se dieron de baja.

In [ ]:
nombres_baja = set(bajas['Nombre Completo'])
asignados_hoy = activos[activos['Fecha Fin'] == 'Activo']
print('Equipos asignados a personas dadas de baja:', asignados_hoy['Colaborador Asignado'].isin(nombres_baja).sum())

Equipos asignados a personas dadas de baja: 9


Lo de la garantía se confirma, porque en la página el campo está vacío en todos los equipos y el Excel ni siquiera exporta una columna de garantía. También se confirma que 70 de los 123 equipos tienen costo en cero, así que la inversión real es mayor a lo que dice el archivo. Además encontré que 9 laptops siguen asignadas a personas que ya no trabajan en la empresa y que la marca se escribe de varias formas, como DELL y Dell.

## Conclusión

Como se puede observar, los hallazgos que encontré a simple vista en la página sí se confirman con los datos, y al revisarlos a detalle salieron algunos más. El problema principal es que mucha información está en pendiente, vacía o con datos de relleno, sobre todo en las asignaciones, que es justo lo que el director necesita para saber quién está en qué proyecto y quién se va a quedar libre. Lo único que está bien son los sueldos, que sí cuadran y permiten saber cuánto cuesta cada colaborador.

Otra cosa importante es que el sistema no guarda nada de dinero de los proyectos, como la tarifa, las horas o el monto, así que por ahora no se puede calcular la utilidad. A partir de esto podemos pensar en una solución para la empresa, que sería definir reglas para capturar bien los datos, como no permitir roles en pendiente ni fechas de relleno, y agregar la información que falta para que después se pueda construir un tablero que le sirva al director para tomar decisiones.